In [2]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()          # reads .env into environment variables
client = Anthropic()   # picks up ANTHROPIC_API_KEY automatically

response = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello, Claude"}],
)
print(response.content[0].text)


Hi there! How can I help you today?


In [3]:
def llm(prompt):
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(block.text for block in response.content if block.type == "text")


In [4]:
llm("Hey,What's up")

'Hey! Not much on my end, just here and ready to chat. How about you? Anything on your mind today, or just killing time?'

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm("I just discovered the course. Can I join now?")
print(answer)


I'd be glad to help, but I don't know which course you mean. Could you tell me its name, or paste the link or description? Once I know what it is, I can tell you what I know about it.

If it's a course I don't have details on, or if enrollment may have changed since my training, the course page or the provider's site will show whether registration is open, any deadlines, and whether it's still taking new students. If you can't find that, the instructor or support team can tell you directly.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""


In [8]:
answer = llm(prompt)
print(answer)


Yes, you can still join. The course accepts new participants, but if you want a certificate, you need to submit your project while submissions are still open.


## COURSE FAQ DATASET

In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [10]:
print(courses_raw)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 482}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 161}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 398}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 73}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 249}]


In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1456

In [12]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [13]:
from minsearch import Index


index= Index(
    text_fields=["questions","section","answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [18]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results


[{'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '96286b4be4',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Why does the content keep changing?',
  'answer': 'If the homework title contains `[DRAFT]`, it means the homework is not ready yet.\n\nThe homework is ready only when both are true:\n\n- The homework form is open on the course management platform.\n- The homework title does not contain `[DRAFT]`.\n\nUntil then, the content can still change. Working on the material or homework in advance is a

In [16]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    
    return index.search(
    question,
    boost_dict=boost_dict,
    filter_dict=filter_dict,
    num_results=5)

In [19]:
search_results = search(question)

print(search_results)

[{'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '96286b4be4', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Homework: Why does the content keep changing?', 'answer': 'If the homework title contains `[DRAFT]`, it means the homework is not ready yet.\n\nThe homework is ready only when both are true:\n\n- The homework form is open on the course management platform.\n- The homework title does not contain `[DRAFT]`.\n\nUntil then, the content can still change. Working on the material or homework in advance is at your own risk, 